# Thử nghiệm và tối ưu mô hình Titanic (Experiments_1)

## Import thư viện

In [25]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split, GridSearchCV, cross_val_score
from sklearn.svm import SVC
from sklearn.metrics import accuracy_score, f1_score, roc_auc_score
import joblib
import yaml
import os

# Định nghĩa ID thử nghiệm
exp_id = '1'

## 1. Mục tiêu thử nghiệm
+ **Mô tả**:
    + Tuning hyperparameter cho SVC (C, gamma) bằng GridSearchCV.
    + Đánh giá mô hình bằng cross-validation (5-fold).
    + Thêm feature Title từ Name để cải thiện score.
    + Lưu model tốt nhất và log kết quả.
+ **Dữ liệu vào**: Từ processed (train_processed.csv).
+ **Kết quả**: Model tốt nhất, log, và submission mới.

## 2. Load dữ liệu và config

In [26]:
# Load train/test processed
train_path = '../data/processed/train_processed.csv'
test_path = '../data/processed/test_processed.csv'
labels_path = '../data/processed/train_labels.csv'

# Kiểm tra file tồn tại
if not os.path.exists(train_path):
    raise FileNotFoundError(f"File {train_path} not found")
if not os.path.exists(labels_path):
    raise FileNotFoundError(f"File {labels_path} not found")
if not os.path.exists(test_path):
    raise FileNotFoundError(f"File {test_path} not found")

df_train = pd.read_csv(train_path)
df_test = pd.read_csv(test_path)
y = pd.read_csv(labels_path)['Survived']

# Load config
config_path = f'../exps/configs/params_{exp_id}.yaml'
if not os.path.exists(config_path):
    raise FileNotFoundError(f"Config file {config_path} not found")
with open(config_path, 'r') as file:
    params = yaml.safe_load(file)
print("Params loaded:", params)

# Split train/val
X = df_train
X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

EmptyDataError: No columns to parse from file

## 3. Feature engineering: Trích Title từ Name
+ Trích Mr, Mrs, Miss,... từ Name trong train/test gốc.
+ Encode Title thành số.

In [ ]:
# Load raw để lấy Name
train_raw = pd.read_csv('../data/raw/train.csv')
test_raw = pd.read_csv('../data/raw/test.csv')

# Trích Title
df_train['Title'] = train_raw['Name'].str.extract(' ([A-Za-z]+)\.', expand=False)
test_raw['Title'] = test_raw['Name'].str.extract(' ([A-Za-z]+)\.', expand=False)

# Nhóm Title hiếm
rare_titles = ['Dr', 'Rev', 'Col', 'Major', 'Capt', 'Sir', 'Lady', 'Countess', 'Jonkheer']
df_train['Title'] = df_train['Title'].replace(rare_titles, 'Rare')
test_raw['Title'] = test_raw['Title'].replace(rare_titles, 'Rare')

# Encode Title
title_mapping = {'Mr': 0, 'Miss': 1, 'Mrs': 2, 'Master': 3, 'Rare': 4}
df_train['Title'] = df_train['Title'].map(title_mapping).fillna(4).astype(int)
test_raw['Title'] = test_raw['Title'].map(title_mapping).fillna(4).astype(int)

# Thêm Title vào X
X['Title'] = df_train['Title']
X_train['Title'] = X_train.index.map(df_train['Title'])
X_val['Title'] = X_val.index.map(df_train['Title'])

## 4. Tuning SVC với GridSearchCV
+ Tìm C, gamma tốt nhất.
+ Đánh giá bằng cross-validation.

In [ ]:
# GridSearchCV cho SVC
svc = SVC(probability=True, random_state=42)
if params is None or 'svc' not in params:
    print("Error: Config params not loaded or missing 'svc'. Using default params.")
    param_grid = {
        'C': [0.1, 1.0, 10.0],
        'gamma': [0.001, 0.01, 0.1, 'scale'],
        'kernel': ['rbf']
    }
else:
    param_grid = params['svc']
grid_search = GridSearchCV(svc, param_grid=param_grid, cv=5, scoring='accuracy', n_jobs=-1)
grid_search.fit(X_train, y_train)

# In best params và score
print("Best parameters:", grid_search.best_params_)
print("Best CV Accuracy:", grid_search.best_score_)

# Đánh giá trên val
best_svc = grid_search.best_estimator_
y_val_pred = best_svc.predict(X_val)
y_val_prob = best_svc.predict_proba(X_val)[:, 1]
print("Validation Accuracy:", accuracy_score(y_val, y_val_pred))
print("Validation F1:", f1_score(y_val, y_val_pred))
print("Validation ROC AUC:", roc_auc_score(y_val, y_val_prob))

# Cross-validation score
cv_scores = cross_val_score(best_svc, X, y, cv=5, scoring='accuracy')
print("5-Fold CV Accuracy:", cv_scores.mean(), "+/-", cv_scores.std())

TypeError: 'NoneType' object is not subscriptable

## 5. Lưu model và log kết quả
+ Lưu best SVC vào saved_models.
+ Ghi log (params, scores) vào logs.

In [ ]:
# Train full data với best params
best_svc.fit(X, y)
joblib.dump(best_svc, f'../exps/saved_models/model_{exp_id}.pkl')

# Lưu log
log = {
    'exp_id': exp_id,
    'best_params': grid_search.best_params_,
    'cv_accuracy': grid_search.best_score_,
    'val_accuracy': accuracy_score(y_val, y_val_pred),
    'val_f1': f1_score(y_val, y_val_pred),
    'val_roc_auc': roc_auc_score(y_val, y_val_prob),
    'cv_mean': cv_scores.mean(),
    'cv_std': cv_scores.std()
}
with open(f'../exps/logs/log_{exp_id}.txt', 'w') as f:
    f.write(str(log))
print(f"Model and log saved: ../exps/saved_models/model_{exp_id}.pkl, ../exps/logs/log_{exp_id}.txt")

## 6. Tạo submission mới
+ Dự đoán trên test với model tuned.

In [ ]:
# Load test processed
test_path = '../data/processed/test_processed.csv'
df_test = pd.read_csv(test_path)
df_test['Title'] = test_raw['Title']

# Predict
predictions = best_svc.predict(df_test).astype(int)

# Tạo submission
test_orig = pd.read_csv('../data/raw/test.csv')
submission = pd.DataFrame({'PassengerId': test_orig['PassengerId'], 'Survived': predictions})
submission.to_csv('../submission_tuned.csv', index=False)
print("Tuned submission created.")

# Kết thúc